In [7]:
%pip install pandas openpyxl

In [9]:
import pandas as pd
import os

# Dicionário oficial mapeando o caminho relativo de cada arquivo 
# para o seu respectivo nome de unidade devidamente padronizado
file_mapping = {
    '../db/ALCANTARA.xlsx': 'Alcântara',
    '../db/Campo grande.xlsx': 'Campo Grande',
    '../db/DUQUE DE CAXIAS.xlsx': 'Duque de Caxias',
    '../db/Iraja.xlsx': 'Irajá',
    '../db/ITAGUAÍ.xlsx': 'Itaguaí',
    '../db/PIABETA.xlsx': 'Piabetá',
    '../db/Santa Cruz.xlsx': 'Santa Cruz',
    '../db/VILAR DOS TELES.xlsx': 'Vilar dos Teles'
}

# Validação rápida para confirmar que todos os arquivos existem antes de consolidar
print("--- VERIFICAÇÃO DE ARQUIVOS ---")
for filepath, unidade in file_mapping.items():
    if os.path.exists(filepath):
        print(f"[OK] Encontrado: {os.path.basename(filepath)} -> Mapeado como: {unidade}")
    else:
        print(f"[ERRO] Arquivo não encontrado: {filepath}")

--- VERIFICAÇÃO DE ARQUIVOS ---
[OK] Encontrado: ALCANTARA.xlsx -> Mapeado como: Alcântara
[OK] Encontrado: Campo grande.xlsx -> Mapeado como: Campo Grande
[OK] Encontrado: DUQUE DE CAXIAS.xlsx -> Mapeado como: Duque de Caxias
[OK] Encontrado: Iraja.xlsx -> Mapeado como: Irajá
[OK] Encontrado: ITAGUAÍ.xlsx -> Mapeado como: Itaguaí
[OK] Encontrado: PIABETA.xlsx -> Mapeado como: Piabetá
[OK] Encontrado: Santa Cruz.xlsx -> Mapeado como: Santa Cruz
[OK] Encontrado: VILAR DOS TELES.xlsx -> Mapeado como: Vilar dos Teles


In [15]:
import os

# Diretório onde estão os arquivos
diretorio_db = '../db/'

# Dicionário de mapeamento: Nome Antigo -> Novo Nome em camelCase limpo
renomeacoes = {
    'ALCANTARA.xlsx': 'alcantara.xlsx',
    'Campo grande.xlsx': 'campoGrande.xlsx',
    'DUQUE DE CAXIAS.xlsx': 'duqueDeCaxias.xlsx',
    'Iraja.xlsx': 'iraja.xlsx',
    'ITAGUAÍ.xlsx': 'itaguai.xlsx',
    'PIABETA.xlsx': 'piabeta.xlsx',
    'Santa Cruz.xlsx': 'santaCruz.xlsx',
    'VILAR DOS TELES.xlsx': 'vilarDosTeles.xlsx'
}

print("--- INICIANDO RENOMEAÇÃO DOS ARQUIVOS ---")
for antigo, novo in renomeacoes.items():
    caminho_antigo = os.path.join(diretorio_db, antigo)
    caminho_novo = os.path.join(diretorio_db, novo)
    
    if os.path.exists(caminho_antigo):
        os.rename(caminho_antigo, caminho_novo)
        print(f"[Sucesso] {antigo}  --->  {novo}")
    else:
        # Se você já tiver executado ou se o arquivo já estiver com o novo nome
        if os.path.exists(caminho_novo):
            print(f"[Info] O arquivo '{novo}' já existe com esse nome.")
        else:
            print(f"[Erro] Arquivo não encontrado: {antigo}")

print("\nProcesso concluído! Verifique a pasta 'db' no seu explorador do VS Code.")

--- INICIANDO RENOMEAÇÃO DOS ARQUIVOS ---
[Sucesso] ALCANTARA.xlsx  --->  alcantara.xlsx
[Info] O arquivo 'campoGrande.xlsx' já existe com esse nome.
[Info] O arquivo 'duqueDeCaxias.xlsx' já existe com esse nome.
[Sucesso] Iraja.xlsx  --->  iraja.xlsx
[Info] O arquivo 'itaguai.xlsx' já existe com esse nome.
[Sucesso] PIABETA.xlsx  --->  piabeta.xlsx
[Info] O arquivo 'santaCruz.xlsx' já existe com esse nome.
[Info] O arquivo 'vilarDosTeles.xlsx' já existe com esse nome.

Processo concluído! Verifique a pasta 'db' no seu explorador do VS Code.


In [16]:
import pandas as pd
import os

# Lista de arquivos em camelCase na pasta ../db/
arquivos_camel_case = [
    '../db/alcantara.xlsx',
    '../db/campoGrande.xlsx',
    '../db/duqueDeCaxias.xlsx',
    '../db/iraja.xlsx',
    '../db/itaguai.xlsx',
    '../db/piabeta.xlsx',
    '../db/santaCruz.xlsx',
    '../db/vilarDosTeles.xlsx'
]

dfs = []

for arquivo in arquivos_camel_case:
    try:
        df = pd.read_excel(arquivo, sheet_name=0)
        df = df.dropna(how='all')
        
        # Padroniza colunas essenciais se faltarem
        if 'Telefone' not in df.columns:
            df['Telefone'] = None
        if 'E-mail' not in df.columns:
            df['E-mail'] = None
            
        # Extrai o nome da unidade limpo com base no nome do arquivo
        nome_unidade = os.path.basename(arquivo).replace('.xlsx', '').title()
        if 'Unidade' not in df.columns:
            df['Unidade'] = nome_unidade
        else:
            df['Unidade'] = df['Unidade'].fillna(nome_unidade)
            
        dfs.append(df)
        print(f"[Lido com sucesso] {os.path.basename(arquivo)}")
    except Exception as e:
        print(f"[Erro] Falha ao ler {arquivo}: {e}")

# Consolidação inicial
df_bruto = pd.concat(dfs, ignore_index=True)
df_bruto = df_bruto.dropna(subset=['Nome completo'])

# --- TRATAMENTO DE TELEFONES MÚLTIPLOS (EXPLODE) ---
registros_tratados = []

for _, row in df_bruto.iterrows():
    nome = str(row['Nome completo']).strip().title()
    email = str(row['E-mail']).strip() if pd.notna(row['E-mail']) else None
    unidade = str(row['Unidade']).strip().title()
    tel_raw = str(row['Telefone']).strip()
    
    # Verifica se há múltiplos telefones separados por barra '/'
    if '/' in tel_raw and tel_raw != 'nan':
        telefones_separados = tel_raw.split('/')
        for t in telefones_separados:
            nova_row = row.copy()
            nova_row['Nome completo'] = nome
            nova_row['Telefone'] = t.strip()
            nova_row['E-mail'] = email
            nova_row['Unidade'] = unidade
            registros_tratados.append(nova_row)
    else:
        nova_row = row.copy()
        nova_row['Nome completo'] = nome
        nova_row['Telefone'] = None if tel_raw == 'nan' else tel_raw
        nova_row['E-mail'] = email
        nova_row['Unidade'] = unidade
        registros_tratados.append(nova_row)

# Cria o DataFrame final tratado e expandido
df_final_tratado = pd.DataFrame(registros_tratados)

print("\n--- RESUMO DA CONSOLIDAÇÃO E TRATAMENTO ---")
print(f"Total de linhas antes da expansão de telefones: {len(df_bruto)}")
print(f"Total de linhas após separar telefones múltiplos: {len(df_final_tratado)}")

# Salva a base unificada e limpa pronta para a próxima etapa
df_final_tratado.to_excel('../db/baseAlunosTratada.xlsx', index=False)
print("\nArquivo salvo com sucesso em '../db/baseAlunosTratada.xlsx'!")

[Lido com sucesso] alcantara.xlsx
[Lido com sucesso] campoGrande.xlsx
[Lido com sucesso] duqueDeCaxias.xlsx
[Lido com sucesso] iraja.xlsx
[Lido com sucesso] itaguai.xlsx
[Lido com sucesso] piabeta.xlsx
[Lido com sucesso] santaCruz.xlsx
[Lido com sucesso] vilarDosTeles.xlsx

--- RESUMO DA CONSOLIDAÇÃO E TRATAMENTO ---
Total de linhas antes da expansão de telefones: 5415
Total de linhas após separar telefones múltiplos: 5685

Arquivo salvo com sucesso em '../db/baseAlunosTratada.xlsx'!


In [19]:
import pandas as pd

# Carrega o arquivo tratado que acabamos de gerar
df_lista_alunos_leads = pd.read_excel('../db/baseAlunosTratada.xlsx')

# Exibe as primeiras 10 linhas em formato de tabela bonita no Jupyter
display(df_lista_alunos_leads.head(10))

# Mostra o total de linhas e colunas atuais
print(f"\nDimensões da base tratada: {df_lista_alunos_leads.shape[0]} linhas e {df_lista_alunos_leads.shape[1]} colunas.")

,Nome completo,Telefone,E-mail,Unidade,Unnamed: 4,ANO DE CONCLUSÃO,Turno,Turma,Dia da Semana
0,Adriana Gomes Da Silva,(21)97040-4576,ricas.g17@gmail.com,Alcantara,1.0,NaN,NaN,NaN,NaN
1,Karine De Jesus Araújo,(21)96694-9024,araujo.ruvy@gmail.com,Alcantara,2.0,NaN,NaN,NaN,NaN
2,Karine De Jesus Araújo,(21)96418-3205,araujo.ruvy@gmail.com,Alcantara,2.0,NaN,NaN,NaN,NaN
3,Lúcia Helena Gomes Costa,(21)99515-3873,luciahelenagomes2018@gmail.com,Alcantara,3.0,NaN,NaN,NaN,NaN
4,Thamirys Conceição De Abreu,(21)99509-3807,thamirysabreu23@gmail.com,Alcantara,4.0,NaN,NaN,NaN,NaN
5,Thamirys Conceição De Abreu,(21)2601-0671,thamirysabreu23@gmail.com,Alcantara,4.0,NaN,NaN,NaN,NaN
6,Gilderlânia Farias Dos Santos,(21)99074-7713,gilderlaniafarias@gmail.com,Alcantara,5.0,NaN,NaN,NaN,NaN
7,Simone Canuto Coelho,(21)97009-5598,simonecanutoc@gmail.com,Alcantara,6.0,NaN,NaN,NaN,NaN
8,Simone Canuto Coelho,(21)3614-2725,simonecanutoc@gmail.com,Alcantara,6.0,NaN,NaN,NaN,NaN
9,Raquel Pereira De Nazareth,(21)98936-3784,raquelkaryna2015@gmail.com,Alcantara,7.0,NaN,NaN,NaN,NaN



Dimensões da base tratada: 5685 linhas e 9 colunas.


In [20]:
import pandas as pd

# Carrega a base tratada atual
df = pd.read_excel('../db/baseAlunosTratada.xlsx')

# Lista de colunas indesejadas para remover
colunas_para_remover = [
    'Unnamed: 4', 
    'ANO DE CONCLUSÃO', 
    'Turno', 
    'Turma', 
    'Dia da Semana', 
    'Unidade_Arquivo'
]

# Remove de forma segura apenas as colunas que existirem no DataFrame
df = df.drop(columns=[col for col in colunas_para_remover if col in df.columns], errors='ignore')

# Salva a base limpa de volta no arquivo
df.to_excel('../db/baseAlunosTratada.xlsx', index=False)

print("--- BASE LIMPA COM SUCESSO ---")
print("Colunas finais:", df.columns.tolist())
print(f"Total de registros na base final: {len(df)}")

--- BASE LIMPA COM SUCESSO ---
Colunas finais: ['Nome completo', 'Telefone', 'E-mail', 'Unidade']
Total de registros na base final: 5685


In [21]:
import pandas as pd

# Carrega o arquivo tratado que acabamos de gerar
df_lista_alunos_leads = pd.read_excel('../db/baseAlunosTratada.xlsx')

# Exibe as primeiras 10 linhas em formato de tabela bonita no Jupyter
display(df_lista_alunos_leads.head(10))

# Mostra o total de linhas e colunas atuais
print(f"\nDimensões da base tratada: {df_lista_alunos_leads.shape[0]} linhas e {df_lista_alunos_leads.shape[1]} colunas.")

,Nome completo,Telefone,E-mail,Unidade
0,Adriana Gomes Da Silva,(21)97040-4576,ricas.g17@gmail.com,Alcantara
1,Karine De Jesus Araújo,(21)96694-9024,araujo.ruvy@gmail.com,Alcantara
2,Karine De Jesus Araújo,(21)96418-3205,araujo.ruvy@gmail.com,Alcantara
3,Lúcia Helena Gomes Costa,(21)99515-3873,luciahelenagomes2018@gmail.com,Alcantara
4,Thamirys Conceição De Abreu,(21)99509-3807,thamirysabreu23@gmail.com,Alcantara
5,Thamirys Conceição De Abreu,(21)2601-0671,thamirysabreu23@gmail.com,Alcantara
6,Gilderlânia Farias Dos Santos,(21)99074-7713,gilderlaniafarias@gmail.com,Alcantara
7,Simone Canuto Coelho,(21)97009-5598,simonecanutoc@gmail.com,Alcantara
8,Simone Canuto Coelho,(21)3614-2725,simonecanutoc@gmail.com,Alcantara
9,Raquel Pereira De Nazareth,(21)98936-3784,raquelkaryna2015@gmail.com,Alcantara



Dimensões da base tratada: 5685 linhas e 4 colunas.


In [22]:
import pandas as pd
import re

# Carrega a base atual tratada
df = pd.read_excel('../db/baseAlunosTratada.xlsx')

def formatar_telefone_whatsapp(tel):
    if pd.isna(tel):
        return None
    
    # Remove tudo o que não for número (parênteses, traços, espaços, etc.)
    digitos = re.sub(r'\D', '', str(tel))
    
    # Se o número tiver 10 ou 11 dígitos (ex: 21970404576), adiciona o DDI '55' do Brasil na frente
    if len(digitos) in [10, 11]:
        digitos = '55' + digitos
        
    return digitos

# Aplica a função de limpeza na coluna de Telefone
df['Telefone_Limpo'] = df['Telefone'].apply(formatar_telefone_whatsapp)

# Substitui a coluna antiga pela coluna limpa padronizada
df['Telefone'] = df['Telefone_Limpo']
df = df.drop(columns=['Telefone_Limpo'])

# Removemos eventuais linhas onde o telefone ficou vazio ou inválido
df = df.dropna(subset=['Telefone'])

# Salva a base definitiva pronta para o disparo
df.to_excel('../db/baseAlunosProntaParaDisparo.xlsx', index=False)

print("--- TELEFONES PADRONIZADOS COM SUCESSO ---")
print(f"Total de contatos válidos prontos para disparo: {len(df)}")
print("\nAmostra dos primeiros telefones formatados:")
print(df[['Nome completo', 'Telefone', 'Unidade']].head(5))

--- TELEFONES PADRONIZADOS COM SUCESSO ---
Total de contatos válidos prontos para disparo: 5293

Amostra dos primeiros telefones formatados:
                 Nome completo       Telefone    Unidade
0       Adriana Gomes Da Silva  5521970404576  Alcantara
1       Karine De Jesus Araújo  5521966949024  Alcantara
2       Karine De Jesus Araújo  5521964183205  Alcantara
3     Lúcia Helena Gomes Costa  5521995153873  Alcantara
4  Thamirys Conceição De Abreu  5521995093807  Alcantara
